# Explore taxonomy RDF — local SPARQL queries

Uses `rdflib` to load the generated TTL bundles locally and run SPARQL queries interactively.  
No server required — edit queries in-place and re-run cells.

Once a query looks good, copy it to **[SPARQLQueries](https://github.com/pathway-lod/SPARQLQueries)** and add it to **[Snorql-UI](https://github.com/pathway-lod/Snorql-UI)**.

**Kernel:** select `plantmetwiki-rdf` (register once with `python -m ipykernel install --user --name plantmetwiki-rdf`).

---

## ⚠️ Limitations — read before writing queries

`rdflib` is a pure-Python in-memory SPARQL engine with **no query optimizer**:

| ✅ Works well | ❌ Avoid — will hang for minutes/hours |
|---|---|
| Simple `SELECT` with 1–2 triple patterns on one graph | **Merging graphs** with `g_a + g_b` — creates massive combined graph |
| `COUNT` / `GROUP BY` on one variable | `FILTER(STRSTARTS(?a, STR(?b)))` over large sets — O(n²) cartesian product |
| Queries on taxonomy bundle (~4 MB, fast) | Joining two triple patterns with no shared variable |
| Properties bundle queries (load separately, ~160 MB) | Queries on merged `g_both` — slow even if individual graphs are fast |

**URI structure** (pathway ≠ DataNode prefix — do NOT use STRSTARTS to link them):
```
Pathway:  http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC1_r<version>
DataNode: http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC1_r<version>/DataNode/<id>
```

**For cross-layer queries** (pathway ↔ DataNode ↔ species), use the **Virtuoso** endpoint. See section 5.

In [36]:
from rdflib import Graph, Namespace, URIRef
import pandas as pd
from pathlib import Path

PREFIXES = """
PREFIX wp:      <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi:    <http://purl.obolibrary.org/obo/NCBITaxon_>
PREFIX pmw:     <http://rdf-plantmetwiki.bioinformatics.nl/vocab/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
"""

def sparql(g: Graph, query: str) -> pd.DataFrame:
    """Run a SPARQL SELECT and return a DataFrame."""
    results = g.query(PREFIXES + query)
    return pd.DataFrame(results, columns=[str(v) for v in results.vars])

print("Ready.")

Ready.


## Load bundles — keep them SEPARATE

**Do not merge** `g_tax + g_prop` — the properties bundle has hundreds of thousands of blank nodes and the combined graph is extremely slow to query in rdflib. Query each graph independently.

In [37]:
VERSION = "plantcyc17.0.0-gpml2021"
bundles = Path("../output/bundles")

# ── Taxonomy extra (~4 MB, fast) ──────────────────────────────────────────────
print("Loading taxonomy extra...")
g_tax = Graph()
g_tax.parse(str(bundles / f"all_gpml_taxonomy_extra-{VERSION}.ttl"), format="turtle")
print(f"  {len(g_tax):,} triples  (all wp:organism statements)")

# ── Properties extra (~160 MB, ~1 min) ───────────────────────────────────────
print("Loading properties extra (this takes ~1 min)...")
g_prop = Graph()
g_prop.parse(str(bundles / f"all_gpml_properties_extra-{VERSION}.ttl"), format="turtle")
print(f"  {len(g_prop):,} triples  (pmw:gpmlProperty blank nodes + pmw:plantcycId)")

Loading taxonomy extra...
  27,628 triples  (all wp:organism statements)
Loading properties extra (this takes ~1 min)...
  2,615,361 triples  (pmw:gpmlProperty blank nodes + pmw:plantcycId)


---
## 1. Pathway IRIs and PlantCyc IDs

Each pathway has a stable IRI (`/pathways/PC{n}_r{version}`) and a PlantCyc ID stored as `pmw:plantcycId`.

In [38]:
# Pathway IRI + PlantCyc ID + identifiers.org source — sample 10
sparql(g_prop, """
SELECT ?pathway_iri ?plantcyc_id ?source
WHERE {
    ?pathway_iri pmw:plantcycId  ?plantcyc_id .
    ?pathway_iri dcterms:source  ?source .
    FILTER(CONTAINS(STR(?pathway_iri), "/pathways/"))
}
LIMIT 10
""")

,pathway_iri,plantcyc_id,source
0,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-6933,https://identifiers.org/plantcyc/PWY-6933
1,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-7158,https://identifiers.org/plantcyc/PWY-7158
2,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-6303,https://identifiers.org/plantcyc/PWY-6303
3,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-6585,https://identifiers.org/plantcyc/PWY-6585
4,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-5080,https://identifiers.org/plantcyc/PWY-5080
5,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-7117,https://identifiers.org/plantcyc/PWY-7117
6,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-5829,https://identifiers.org/plantcyc/PWY-5829
7,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-6213,https://identifiers.org/plantcyc/PWY-6213
8,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-1186,https://identifiers.org/plantcyc/PWY-1186
9,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-5350,https://identifiers.org/plantcyc/PWY-5350


In [39]:
# Print full pathway IRIs (pandas truncates them by default)
df = sparql(g_prop, """
SELECT ?pathway_iri ?plantcyc_id
WHERE {
    ?pathway_iri pmw:plantcycId ?plantcyc_id .
    FILTER(CONTAINS(STR(?pathway_iri), "/pathways/"))
}
LIMIT 10
""")
for _, row in df.iterrows():
    print(f"{row['plantcyc_id']:<20}  {row['pathway_iri']}")

PWY-6933              http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC1_r17.0.0_20260515134008
PWY-7158              http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC10_r17.0.0_20260515134008
PWY-6303              http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC100_r17.0.0_20260515134008
PWY-6585              http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC1000_r17.0.0_20260515134008
PWY-5080              http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC1001_r17.0.0_20260515134008
PWY-7117              http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC1002_r17.0.0_20260515134008
PWY-5829              http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC1003_r17.0.0_20260515134008
PWY-6213              http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC1004_r17.0.0_20260515134008
PWY-1186              http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC1005_r17.0.0_20260515134008
PWY-5350              http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC1006

In [40]:
# Total pathways with a PlantCyc ID — expected: 1162
sparql(g_prop, """
SELECT (COUNT(DISTINCT ?iri) AS ?pathways_with_plantcyc_id)
WHERE {
    ?iri pmw:plantcycId ?id .
    FILTER(CONTAINS(STR(?iri), "/pathways/"))
}
""")

,pathways_with_plantcyc_id
0,2478


---
## 2. Verify taxonomy and properties use the same IRIs

Both bundles use the same pathway IRI as subject — they are implicitly linked by shared URI.  
We verify this in Python (fast set comparison) rather than a slow SPARQL join.

In [41]:
# Spot-check: verify sample pathway IRIs from taxonomy-extra appear in the core bundle
# Uses text search (no full load of the 300 MB core file)
import re

core_file = Path(f"../output/bundles/all-{VERSION}.ttl")
sample_iris = sorted(tax_pathways)[:10]

print(f"Checking {len(sample_iris)} sample pathway IRIs against {core_file.name} ...")
core_text = core_file.read_text(encoding="utf-8")

all_found = True
for iri in sample_iris:
    short = iri.split("/")[-1]
    found = f"<{iri}>" in core_text
    print(f"  {'✓' if found else '✗  ← MISSING'} {short}")
    if not found:
        all_found = False

print()
print("✅ All sample IRIs found in core bundle" if all_found else "❌ Some IRIs missing — check URI generation")

Checking 10 sample pathway IRIs against all-plantcyc17.0.0-gpml2021.ttl ...
  ✓ PC1000_r17.0.0_20260515134008
  ✓ PC1001_r17.0.0_20260515134008
  ✓ PC1002_r17.0.0_20260515134008
  ✓ PC1003_r17.0.0_20260515134008
  ✓ PC1004_r17.0.0_20260515134008
  ✓ PC1005_r17.0.0_20260515134008
  ✓ PC1006_r17.0.0_20260515134008
  ✓ PC1007_r17.0.0_20260515134008
  ✓ PC1008_r17.0.0_20260515134008
  ✓ PC1009_r17.0.0_20260515134008

✅ All sample IRIs found in core bundle


In [42]:
from rdflib import URIRef

WP_ORGANISM  = URIRef("http://vocabularies.wikipathways.org/wp#organism")
PMW_ID       = URIRef("http://rdf-plantmetwiki.bioinformatics.nl/vocab/plantcycId")
VIRIDIPLANTAE = URIRef("http://purl.obolibrary.org/obo/NCBITaxon_33090")

# Pathway IRIs in taxonomy bundle (those with Viridiplantae + /pathways/ in URI)
tax_pathways  = {str(s) for s, p, o in g_tax
                 if p == WP_ORGANISM and o == VIRIDIPLANTAE
                 and "/pathways/" in str(s)}

# Pathway IRIs in properties bundle
prop_pathways = {str(s) for s, p, o in g_prop
                 if p == PMW_ID
                 and "/pathways/" in str(s)}

print(f"Pathway IRIs in taxonomy extra:   {len(tax_pathways):,}")
print(f"Pathway IRIs in properties extra: {len(prop_pathways):,}")
print(f"IRIs in both (shared):            {len(tax_pathways & prop_pathways):,}")
print(f"Only in taxonomy:                 {len(tax_pathways - prop_pathways):,}")
print(f"Only in properties:               {len(prop_pathways - tax_pathways):,}")

Pathway IRIs in taxonomy extra:   2,478
Pathway IRIs in properties extra: 2,478
IRIs in both (shared):            2,478
Only in taxonomy:                 0
Only in properties:               0


---
## 3. Explore GPML properties

All `<Property key="..." value="...">` elements from the GPML files are stored as blank nodes:
```turtle
?subject pmw:gpmlProperty [ pmw:key "..." ; pmw:value "..." ] .
```

**Note on the `Organism` property:** the current GPML files carry `organism="Viridiplantae"` on the `<Pathway>` element (the XSD-valid single-species annotation). However, the original per-pathway species list from PlantCyc (e.g. `"Arabidopsis thaliana, Glycine max, ..."`) is **intentionally preserved** as a `<Property key="Organism">` so no information is lost. It ends up in this bundle as a `pmw:gpmlProperty` blank node with key `"Organism"`, and can be queried as shown below. This is distinct from the structured `wp:organism` triples in the taxonomy-extra bundle — those carry machine-readable NCBI taxon IDs on individual DataNodes.

In [43]:
# All unique property keys and how often they appear — top 20
sparql(g_prop, """
SELECT ?key (COUNT(?key) AS ?count)
WHERE {
    ?subject pmw:gpmlProperty ?bn .
    ?bn pmw:key ?key .
}
GROUP BY ?key
ORDER BY DESC(?count)
LIMIT 20
""")

,key,count
0,InstanceNameTemplate,54578
1,UniqueID,51528
2,Synonym_1,27711
3,Gibbs0,27049
4,Osmolarity,22564
5,Smiles,21976
6,NonStandardInchi,21563
7,MolecularWeight,20813
8,ChemicalFormula,20698
9,MonoisotopicMw,20695


In [44]:
# Original per-pathway species list from PlantCyc
# Preserved intentionally as Property key="Organism" so the multi-species
# information is not lost now that pathway organism attribute = "Viridiplantae"
sparql(g_prop, """
SELECT ?pathway_iri ?original_species
WHERE {
    ?pathway_iri pmw:gpmlProperty ?bn .
    ?bn pmw:key   "Organism" ;
        pmw:value ?original_species .
    FILTER(CONTAINS(STR(?pathway_iri), "/pathways/"))
}
LIMIT 10
""")

,pathway_iri,original_species
0,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Brassica oleracea, Brassica juncea, Astragalus..."
1,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Chlamydomonas reinhardtii, Physcomitrium paten..."
2,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Populus trichocarpa, Arabidopsis thaliana"
3,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Lycopersicon hirsutum, Solanum, Solanum habroc..."
4,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Arabidopsis thaliana, Brassica napus, Limnanth..."
5,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Zea mays, Megathyrsus maximus, Urochloa panico..."
6,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Vitis vinifera, Zea mays, Cinnamomum tenuipile..."
7,http://rdf-plantmetwiki.bioinformatics.nl/path...,Arabidopsis thaliana
8,http://rdf-plantmetwiki.bioinformatics.nl/path...,Arabidopsis thaliana
9,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Arabidopsis thaliana, Arabidopsis thaliana, Gl..."


In [45]:
# DataNode UniqueID values (PlantCyc gene/protein/compound IDs)
sparql(g_prop, """
SELECT ?datanode ?plantcyc_id
WHERE {
    ?datanode pmw:gpmlProperty ?bn .
    ?bn pmw:key   "UniqueID" ;
        pmw:value ?plantcyc_id .
    FILTER(CONTAINS(STR(?datanode), "/DataNode/"))
}
LIMIT 10
""")

,datanode,plantcyc_id
0,http://rdf-plantmetwiki.bioinformatics.nl/Path...,G-11674
1,http://rdf-plantmetwiki.bioinformatics.nl/Path...,G-11675
2,http://rdf-plantmetwiki.bioinformatics.nl/Path...,MONOMER-15249
3,http://rdf-plantmetwiki.bioinformatics.nl/Path...,MONOMER-15250
4,http://rdf-plantmetwiki.bioinformatics.nl/Path...,CPD-12006
5,http://rdf-plantmetwiki.bioinformatics.nl/Path...,ADENOSYL-HOMO-CYS
6,http://rdf-plantmetwiki.bioinformatics.nl/Path...,L-SELENOCYSTEINE
7,http://rdf-plantmetwiki.bioinformatics.nl/Path...,Acceptor
8,http://rdf-plantmetwiki.bioinformatics.nl/Path...,WATER
9,http://rdf-plantmetwiki.bioinformatics.nl/Path...,AMMONIUM


---
## 4. Taxonomy queries

All run on the small `g_tax` graph only.

In [46]:
# Count resources with Viridiplantae — expected 2478 (1162 pathways + 1316 reactions)
sparql(g_tax, """
SELECT (COUNT(DISTINCT ?resource) AS ?resources_with_viridiplantae)
WHERE { ?resource wp:organism ncbi:33090 . }
""")

,resources_with_viridiplantae
0,2478


In [47]:
# Species distribution across DataNode URIs — top 20
sparql(g_tax, """
SELECT ?species (COUNT(DISTINCT ?node) AS ?node_count)
WHERE {
    ?node wp:organism ?species .
    FILTER(?species != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
}
GROUP BY ?species
ORDER BY DESC(?node_count)
LIMIT 20
""")

,species,node_count
0,http://purl.obolibrary.org/obo/NCBITaxon_3702,7719
1,http://purl.obolibrary.org/obo/NCBITaxon_3847,1078
2,http://purl.obolibrary.org/obo/NCBITaxon_34305,926
3,http://purl.obolibrary.org/obo/NCBITaxon_3055,524
4,http://purl.obolibrary.org/obo/NCBITaxon_381124,513
5,http://purl.obolibrary.org/obo/NCBITaxon_4081,421
6,http://purl.obolibrary.org/obo/NCBITaxon_4113,305
7,http://purl.obolibrary.org/obo/NCBITaxon_3888,236
8,http://purl.obolibrary.org/obo/NCBITaxon_46611,230
9,http://purl.obolibrary.org/obo/NCBITaxon_39947,214


In [48]:
# Sample DataNode + biological entity URIs for Arabidopsis thaliana (ncbi:3702)
sparql(g_tax, """
SELECT ?uri
WHERE { ?uri wp:organism ncbi:3702 . }
LIMIT 10
""")

,uri
0,http://rdf-plantmetwiki.bioinformatics.nl/Path...
1,https://identifiers.org/tair.name/AT3G10870
2,http://rdf-plantmetwiki.bioinformatics.nl/Path...
3,https://identifiers.org/tair.name/AT5G55250
4,http://rdf-plantmetwiki.bioinformatics.nl/Path...
5,https://identifiers.org/uniprot/Q9FLN8
6,http://rdf-plantmetwiki.bioinformatics.nl/Path...
7,https://identifiers.org/uniprot/Q9SG92
8,http://rdf-plantmetwiki.bioinformatics.nl/Path...
9,https://identifiers.org/tair.name/AT1G68530


In [49]:
# UniProt URIs with species annotation
sparql(g_tax, """
SELECT ?entity ?species
WHERE {
    ?entity wp:organism ?species .
    FILTER(STRSTARTS(STR(?entity), "https://identifiers.org/uniprot/"))
}
LIMIT 10
""")

,entity,species
0,https://identifiers.org/uniprot/Q4VNK0,http://purl.obolibrary.org/obo/NCBITaxon_36774
1,https://identifiers.org/uniprot/A4ZGQ8,http://purl.obolibrary.org/obo/NCBITaxon_36774
2,https://identifiers.org/uniprot/E5KBU4,http://purl.obolibrary.org/obo/NCBITaxon_3218
3,https://identifiers.org/uniprot/A5A8G0,http://purl.obolibrary.org/obo/NCBITaxon_3218
4,https://identifiers.org/uniprot/A8IT23,http://purl.obolibrary.org/obo/NCBITaxon_3055
5,https://identifiers.org/uniprot/A8HQD7,http://purl.obolibrary.org/obo/NCBITaxon_3055
6,https://identifiers.org/uniprot/A8JFB0,http://purl.obolibrary.org/obo/NCBITaxon_3055
7,https://identifiers.org/uniprot/A8JI07,http://purl.obolibrary.org/obo/NCBITaxon_3055
8,https://identifiers.org/uniprot/Q8VZZ0,http://purl.obolibrary.org/obo/NCBITaxon_3055
9,https://identifiers.org/uniprot/Q9FYU1,http://purl.obolibrary.org/obo/NCBITaxon_3055


---
## 5. Cross-layer queries → Virtuoso only

These need `wp:isPartOf` from the core bundle. Run on the **Virtuoso endpoint** using named graphs.

```sparql
# Pathways containing at least one Arabidopsis thaliana DataNode
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi: <http://purl.obolibrary.org/obo/NCBITaxon_>

SELECT ?pathway (COUNT(DISTINCT ?node) AS ?arabidopsis_nodes)
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways> {
      ?node wp:isPartOf ?pathway .
  }
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-taxonomy-extra> {
      ?node wp:organism ncbi:3702 .
  }
}
GROUP BY ?pathway
ORDER BY DESC(?arabidopsis_nodes)
LIMIT 20
```

```sparql
# Pathway IRI + PlantCyc ID + Viridiplantae annotation
PREFIX wp:      <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi:    <http://purl.obolibrary.org/obo/NCBITaxon_>
PREFIX pmw:     <http://rdf-plantmetwiki.bioinformatics.nl/vocab/>

SELECT ?pathway_iri ?plantcyc_id ?organism
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-properties-extra> {
      ?pathway_iri pmw:plantcycId ?plantcyc_id .
  }
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-taxonomy-extra> {
      ?pathway_iri wp:organism ?organism .
  }
}
LIMIT 20
```

---
## 6. Sandbox

Use `g_tax` or `g_prop` — **not both together**. When a query is validated, copy it to **[SPARQLQueries](https://github.com/pathway-lod/SPARQLQueries)**.

In [50]:
# Replace g_tax with g_prop if querying properties
sparql(g_tax, """
SELECT *
WHERE {
    # ← write your query here
}
LIMIT 20
""")

""
